# PHM2012 RUL 复现：FFT + CBAM-CNN-LSTM

参考论文：Remaining Useful Life Prediction of Rolling Bearings Based on CBAM-CNN-LSTM。

- 本地 PDF：`../../docs/papers/2025-Sun-PHM2012-RUL-CBAM-CNN-LSTM.pdf`
- DOI：https://doi.org/10.3390/s25020554

流程与 demo 保持一致：配置路径、读取/缓存特征、展示特征图与热力图、构建序列数据集、调用框架 `BaseTrainer` 训练、评估并可视化预测结果。

## 1. 环境与路径

In [ ]:
from pathlib import Path
import random
import time

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn
from torch.utils.data import DataLoader

from phm.data import Dataset
from phm.data.paper import (
    PHM2012_FULL_TEST_BEARINGS,
    PHM2012_LEARNING_BEARINGS,
    SequenceFeatureDataset,
    build_phm2012_rul_feature_cache,
    fit_feature_standardizer,
    make_sequence_index,
    load_training_artifacts,
    read_phm2012_acc_file,
    save_training_artifacts,
)
from phm.engine.trainer.BaseTrainer import BaseTrainer
from phm.model.paper import PaperCBAMCNNLSTMRegressor
from phm.util.Device import select_torch_device


In [ ]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "data" / "loader_roots").exists():
            return path
    raise FileNotFoundError("Could not find data/loader_roots from the current working directory.")

PROJECT_ROOT = find_project_root()
PHM2012_ROOT = PROJECT_ROOT / "data" / "loader_roots" / "phm2012"
CACHE_PATH = PROJECT_ROOT / "cache" / "paper_features" / "phm2012_rul_fft256_full.npz"
CHECKPOINT_PATH = PROJECT_ROOT / "cache" / "paper_checkpoints" / "phm2012_paper_cbam_cnn_lstm_feature_enhanced.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
FFT_BINS = 256
SEQUENCE_LENGTH = 32
SEQUENCE_STEP = 1
BATCH_SIZE = 128
MAX_EPOCHS = 200
LEARNING_RATE = 7e-4
WEIGHT_DECAY = 1e-4
FORCE_REBUILD_CACHE = False
RUN_TRAINING = True
PAPER_TEST_BEARINGS = ("Bearing1_3", "Bearing1_4", "Bearing1_5", "Bearing1_7", "Bearing2_3", "Bearing2_6")
SAMPLING_RATE = 25_600

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = select_torch_device()
print(f"project_root={PROJECT_ROOT}")
print(f"device={device}")


## 2. 全量文件级特征缓存

In [ ]:
start_time = time.time()
feature_data = build_phm2012_rul_feature_cache(
    PHM2012_ROOT,
    CACHE_PATH,
    fft_bins=FFT_BINS,
    include_handcrafted=True,
    force=FORCE_REBUILD_CACHE,
)
features = feature_data["features"]
targets = feature_data["targets"]
ranges = feature_data["ranges"]
print(f"features={features.shape}, targets={targets.shape}")
print(f"bearings={len(ranges)}, cache={CACHE_PATH}")
print(f"elapsed={time.time() - start_time:.1f}s")


## 3. 特征图与热力图

In [ ]:
def zscore_for_plot(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - x.mean(axis=0, keepdims=True)) / (x.std(axis=0, keepdims=True) + 1e-6)

bearing_name = "Bearing1_1"
first_file = sorted((PHM2012_ROOT / "Learning_set" / bearing_name).glob("acc_*.csv"))[0]
first_signal = read_phm2012_acc_file(first_file, bearing_name)[:, 0]
time_axis_seconds = np.arange(first_signal.shape[0]) / SAMPLING_RATE
fft_axis = np.fft.rfftfreq(first_signal.shape[0], d=1 / SAMPLING_RATE)[1: FFT_BINS + 1]
fft_magnitude = np.log1p(np.abs(np.fft.rfft(first_signal * np.hanning(first_signal.shape[0]))))[1: FFT_BINS + 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(time_axis_seconds, first_signal)
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].set_title(f"{bearing_name} horizontal signal")
axes[0].grid(alpha=0.3)
axes[1].plot(fft_axis, fft_magnitude)
axes[1].set_xlabel("Frequency (Hz)")
axes[1].set_ylabel("Log magnitude")
axes[1].set_title(f"{bearing_name} FFT magnitude")
axes[1].grid(alpha=0.3)
plt.suptitle("Paper Figure 10 style preprocessing view")
plt.tight_layout()
plt.show()

start, end = ranges[bearing_name]
bearing_features = features[start:end]
stride = max(len(bearing_features) // 240, 1)
heat = zscore_for_plot(bearing_features[::stride, :96]).T
plt.figure(figsize=(10, 5))
sns.heatmap(heat, cmap="mako", center=0, cbar_kws={"label": "z-score"})
plt.xlabel("Sampled file index")
plt.ylabel("FFT feature index")
plt.title(f"{bearing_name} FFT feature heatmap")
plt.show()


## 4. 构建序列数据集

In [ ]:
train_val_windows, _ = make_sequence_index(
    ranges,
    sequence_length=SEQUENCE_LENGTH,
    sequence_step=SEQUENCE_STEP,
    bearings=PHM2012_LEARNING_BEARINGS,
)
test_windows, _ = make_sequence_index(
    ranges,
    sequence_length=SEQUENCE_LENGTH,
    sequence_step=SEQUENCE_STEP,
    bearings=PHM2012_FULL_TEST_BEARINGS,
)

rng = np.random.default_rng(SEED)
order = rng.permutation(len(train_val_windows))
val_size = max(int(len(order) * 0.15), 1)
val_windows = train_val_windows[order[:val_size]]
train_windows = train_val_windows[order[val_size:]]

mean, std = fit_feature_standardizer(features, train_windows)
train_dataset = SequenceFeatureDataset(features, targets, train_windows, mean=mean, std=std)
val_dataset = SequenceFeatureDataset(features, targets, val_windows, mean=mean, std=std)
test_dataset = SequenceFeatureDataset(features, targets, test_windows, mean=mean, std=std)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"train_windows={len(train_dataset)}, val_windows={len(val_dataset)}, test_windows={len(test_dataset)}")
print(f"input_dim={features.shape[1]}")

## 5. 调用框架 Trainer 训练

In [ ]:
model = PaperCBAMCNNLSTMRegressor(
    input_dim=features.shape[1],
    lstm_hidden=160,
    lstm_layers=2,
    cbam_reduction=16,
    cbam_kernel_size=7,
    dropout=0.15,
)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
trainer = BaseTrainer(config={
    "device": device,
    "epochs": MAX_EPOCHS,
    "training_note": "enhanced full training with handcrafted degradation features",
    "batch_size": BATCH_SIZE,
    "criterion": criterion,
    "optimizer": optimizer,
    "data_loader": train_loader,
    "grad_clip_norm": 1.0,
})

if RUN_TRAINING:
    loss_history = trainer(model, Dataset(name="PHM2012 enhanced CBAM-CNN-LSTM train"))
    save_training_artifacts(
        model,
        CHECKPOINT_PATH,
        mean=mean,
        std=std,
        config={
            "model": "PaperCBAMCNNLSTMRegressor",
            "fft_bins": FFT_BINS,
            "include_handcrafted": True,
            "sequence_length": SEQUENCE_LENGTH,
            "sequence_step": SEQUENCE_STEP,
            "epochs": MAX_EPOCHS,
    "training_note": "enhanced full training with handcrafted degradation features",
            "loss": "MSELoss",
            "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
        },
    )
else:
    loss_history = {}
    print("RUN_TRAINING=False, skip training")


## 6. 验证集与测试集评估

In [ ]:
def regression_metrics(labels, predictions):
    labels = labels.reshape(-1)
    predictions = predictions.reshape(-1)
    mse = mean_squared_error(labels, predictions)
    return {
        "mse": float(mse),
        "rmse": float(mse ** 0.5),
        "mae": float(mean_absolute_error(labels, predictions)),
        "r2": float(r2_score(labels, predictions)),
    }


def predict_loader(model, loader, device):
    model.to(device)
    model.eval()
    predictions = []
    labels = []
    losses = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.float32)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            predictions.append(y_hat.cpu().numpy())
            labels.append(y.cpu().numpy())
            losses.append(loss.item() * x.size(0))
    predictions = np.vstack(predictions)
    labels = np.vstack(labels)
    metrics = regression_metrics(labels, predictions)
    metrics["loss"] = float(sum(losses) / len(labels))
    return metrics, predictions, labels

if CHECKPOINT_PATH.exists():
    load_training_artifacts(model, CHECKPOINT_PATH, map_location=device)

val_metrics, val_pred, val_true = predict_loader(model, val_loader, device)
test_metrics, test_pred, test_true = predict_loader(model, test_loader, device)
print(
    f"val_loss={val_metrics['loss']:.6f}, val_mse={val_metrics['mse']:.6f}, "
    f"val_rmse={val_metrics['rmse']:.4f}, val_mae={val_metrics['mae']:.4f}, val_r2={val_metrics['r2']:.4f}"
)
print(
    f"test_loss={test_metrics['loss']:.6f}, test_mse={test_metrics['mse']:.6f}, "
    f"test_rmse={test_metrics['rmse']:.4f}, test_mae={test_metrics['mae']:.4f}, test_r2={test_metrics['r2']:.4f}"
)

paper_reference = pd.DataFrame(
    {
        "paper_mse": [0.0047, 0.0077, 0.0198, 0.0137, 0.0223, 0.0085],
        "paper_rmse": [0.069, 0.086, 0.141, 0.117, 0.148, 0.088],
        "paper_mae": [0.037, 0.060, 0.091, 0.078, 0.112, 0.069],
        "paper_r2": [0.943, 0.910, 0.762, 0.836, 0.737, 0.906],
    },
    index=PAPER_TEST_BEARINGS,
)
paper_reference.index.name = "bearing"

bearing_rows = []
predictions_by_bearing = {}
for bearing in PAPER_TEST_BEARINGS:
    bearing_windows, _ = make_sequence_index(
        ranges,
        sequence_length=SEQUENCE_LENGTH,
        sequence_step=SEQUENCE_STEP,
        bearings=[bearing],
    )
    bearing_dataset = SequenceFeatureDataset(features, targets, bearing_windows, mean=mean, std=std)
    bearing_loader = DataLoader(bearing_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    metrics, pred, true = predict_loader(model, bearing_loader, device)
    bearing_rows.append({"bearing": bearing, "windows": len(bearing_dataset), **metrics})
    predictions_by_bearing[bearing] = {"pred": pred.reshape(-1), "true": true.reshape(-1)}

mainline_metrics = pd.DataFrame(bearing_rows).set_index("bearing")
mainline_comparison = mainline_metrics.join(paper_reference)
for metric in ("mse", "rmse", "mae", "r2"):
    mainline_comparison[f"delta_{metric}"] = mainline_comparison[metric] - mainline_comparison[f"paper_{metric}"]

display(mainline_comparison.round(6))


## 7. 训练曲线与结果热力图

In [ ]:
train_losses = next(iter(loss_history.values()), []) if loss_history else []
if train_losses:
    plt.figure(figsize=(7, 4))
    plt.plot(np.arange(1, len(train_losses) + 1), train_losses)
    plt.xlabel("Epoch")
    plt.ylabel("MSE loss")
    plt.title("Paper mainline training curve")
    plt.grid(alpha=0.3)
    plt.show()

plt.figure(figsize=(10, 4))
plot_metrics = mainline_metrics[["mse", "rmse", "mae", "r2"]]
sns.heatmap(plot_metrics, annot=True, fmt=".4f", cmap="viridis", cbar_kws={"label": "Metric value"})
plt.title("Paper Table 3 metrics reproduced with the mainline model")
plt.xlabel("Metric")
plt.ylabel("Test bearing")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, metric in zip(axes, ["mse", "rmse", "mae"]):
    values = mainline_comparison[[metric, f"paper_{metric}"]].rename(columns={metric: "ours", f"paper_{metric}": "paper"})
    values.plot(kind="bar", ax=ax)
    ax.set_title(metric.upper())
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.3)
plt.suptitle("Mainline metrics vs. paper Table 3")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, bearing in zip(axes, ["Bearing1_3", "Bearing2_6"]):
    series = predictions_by_bearing[bearing]
    x = np.arange(len(series["true"]))
    ax.plot(x, series["true"], label="true", linewidth=2)
    ax.plot(x, series["pred"], label="CBAM-CNN-LSTM", linewidth=1.8)
    ax.set_title(f"{bearing} RUL prediction")
    ax.set_xlabel("Test window")
    ax.set_ylabel("Normalized RUL")
    ax.grid(alpha=0.3)
    ax.legend()
plt.suptitle("PHM2012 RUL prediction curves after enhanced training")
plt.tight_layout()
plt.show()

comparison = np.vstack([test_true[:300].reshape(-1), test_pred[:300].reshape(-1)])
plt.figure(figsize=(10, 2.4))
sns.heatmap(comparison, cmap="viridis", vmin=0, vmax=1, cbar_kws={"label": "Normalized RUL"})
plt.yticks([0.5, 1.5], ["true", "pred"], rotation=0)
plt.xlabel("Test window")
plt.title("Overall test RUL prediction heatmap")
plt.show()
